# Lab 3.2: Data Preprocessing & Patch Extraction

**Part of the Iceland ML Course: Sentinel-2 Classification Project**

This notebook demonstrates the preprocessing workflow for Sentinel-2 satellite imagery and extraction of training patches for land classification tasks.

---

## Project Milestone Overview

| Lab | Milestone | Status |
|-----|-----------|--------|
| Lab 1 | HPC Access Setup | ✅ Previous |
| Lab 2 | Jupyter-JSC & Git | ✅ Previous |
| Lab 3.1 | Data Download (Copernicus API) | ✅ Previous |
| Lab 3.2 | **Data Preprocessing & Patch Extraction** | 🔄 **Current** |
| Lab 4 | Understanding Transformers | ⬜ Next |
| Lab 4.1 | Training on Sentinel-2 Data | ⬜ Next |
| Lab 5 | Distributed Training (Multi-GPU) | ⬜ Next |
| Lab 6 | Validation & Performance Metrics | ⬜ Next |
| Lab 7 | Foundation Models & TerraToRCH | ⬜ Final |

---

## What You'll Learn

By the end of this lab, you will:
- Read and inspect geospatial raster files using GDAL
- Extract Sentinel-2 bands from SAFE format archives
- Convert Sentinel-2 data to GeoTIFF format
- Work with ground truth point data (LUCAS)
- Extract 3x3 patches around ground truth points
- Prepare training data for model development

## Quick Start

This lab focuses on the **Data Preprocessing & Patch Extraction** phase:

```
Data Download from Copernicus (Lab 3.1) ✅
    ↓
Data Preprocessing & Patch Extraction (Lab 3.2) ← You are here
    ↓
Model Training (Lab 4+)
```

---

## Overview

This lab consists of:
1. **Reading Raster Data**: Opening and inspecting geospatial files with GDAL
2. **Processing Sentinel-2 SAFE Archives**: Extracting bands and converting to GeoTIFF
3. **Working with Ground Truth Data**: Loading and filtering LUCAS points
4. **Extracting Training Patches**: Creating 3x3 patches around ground truth locations
5. **Saving Training Data**: Preparing data for model training

## Part 1: Working with Geospatial Raster Data

Learn how to read and inspect raster files using GDAL before preprocessing.

In [ ]:
from osgeo import gdal
import os
import numpy as np
import json

### Opening and Inspecting Raster Files

GDAL (Geospatial Data Abstraction Library) is the standard tool for reading geospatial data.

In [ ]:
# Open a GeoTIFF raster file (modify path to your data)
rasterPath = "/p/scratch/training2600/your_username/sentinel2_geotiff/example_tile.tif"

# Open with GDAL (0 = read-only mode)
ds = gdal.Open(rasterPath, 0)

if ds is None:
    print("❌ Could not open raster file. Check the path.")
else:
    # Read first band
    band_ds = ds.GetRasterBand(1)
    data = band_ds.ReadAsArray()
    
    print(f"✓ Raster loaded successfully")
    print(f"  Dimensions: {data.shape}")
    print(f"  Data type: {data.dtype}")
    print(f"  Number of bands: {ds.RasterCount}")

### Inspecting Raster Metadata

In [ ]:
print(data.shape)

In [ ]:
# Examine a small subset of data
if ds is not None:
    print("Sample pixel values (2000:2002, 2000:2002):")
    print(data[2000:2002, 2000:2002])

### Getting Raster Metadata and Projection Information

In [ ]:
# Get detailed metadata
if ds is not None:
    metadata = gdal.Info(ds)
    print("Raster Metadata:")
    print(metadata[:1000])  # Print first 1000 characters

### Extract Geographic Extent (Bounding Box)

In [ ]:
# Get metadata in JSON format for easier parsing
if ds is not None:
    info = json.loads(gdal.Info(ds, format='json'))
    
    # Extract coordinate reference system
    crs = info.get('coordinateSystem', {}).get('wkt', 'Unknown')
    print(f"Coordinate Reference System:")
    print(crs[:200])  # Print first 200 chars
    
    # Extract WGS84 extent (lat/lon bounds)
    if 'wgs84Extent' in info:
        wgs84Extent = info['wgs84Extent']
        print(f"\nWGS84 Extent (Lat/Lon):")
        print(json.dumps(wgs84Extent, indent=2))

---

## Part 2: Processing Sentinel-2 SAFE Archives

Extract Sentinel-2 bands from downloaded SAFE format and convert to GeoTIFF.

### Understanding Sentinel-2 SAFE Format

Sentinel-2 data from Copernicus comes in **SAFE** format:
- `.SAFE` directory containing multiple folders
- `IMG_DATA/` contains band files as JPEG2000 (`.jp2`)
- Bands at different resolutions: 10m, 20m, 60m
- We'll extract key bands and stack them into a single GeoTIFF

In [ ]:
# Example: Checking CRS of a Sentinel-2 band
# This helps ensure CORINE and S2 data are in the same projection

s2_band_path = "/p/scratch/training2600/your_username/sentinel2_extracted/S2A_MSIL2A_*.SAFE/GRANULE/*/IMG_DATA/R10m/*_B02_10m.jp2"

# Open a Sentinel-2 band (B02 - Blue, 10m resolution)
try:
    # Use glob to find the file
    import glob
    matching_files = glob.glob(s2_band_path)
    
    if matching_files:
        s2_path = matching_files[0]
        s2 = gdal.Open(s2_path, 0)
        
        if s2:
            crs = s2.GetProjection()
            print("Sentinel-2 Coordinate Reference System:")
            print(crs[:300])
        else:
            print("❌ Could not open Sentinel-2 band")
    else:
        print("⚠ No Sentinel-2 files found. Adjust the path pattern.")
except Exception as e:
    print(f"⚠ Error: {e}")
    print("Note: Update the path to match your downloaded Sentinel-2 data")

---

## Part 3: Aligning CORINE Land Cover with Sentinel-2

Reproject and extract CORINE map to match Sentinel-2 tile geometry.

In [ ]:
# Step 1: Reproject CORINE to match Sentinel-2 CRS

corine_input = "/p/scratch/training2600/your_username/corine_data/U2018_CLC2018_V2020_20u1.tif"
corine_reproj = "/p/scratch/training2600/your_username/corine_data/corine_reproj_temp.tif"

# Use the CRS from your Sentinel-2 tile
# Example: UTM Zone 32N for Iceland
target_crs = "EPSG:32628"  # Update based on your S2 tile

try:
    # Reproject using gdal.Warp
    gdal.Warp(
        corine_reproj,
        corine_input,
        dstSRS=target_crs,
        resampleAlg='nearest'  # Use nearest neighbor for categorical data
    )
    print(f"✓ CORINE reprojected to {target_crs}")
except Exception as e:
    print(f"❌ Reprojection failed: {e}")
    print("Note: Ensure CORINE file exists at the specified path")

### Step 2: Extract Tile Extent from Sentinel-2

In [ ]:
# Get geographic extent from Sentinel-2 tile for clipping CORINE

s2_geotiff = "/p/scratch/training2600/your_username/sentinel2_geotiff/example_tile.tif"

try:
    s2_ds = gdal.Open(s2_geotiff, 0)
    
    if s2_ds:
        # Get geotransform (ulx, xres, xskew, uly, yskew, yres)
        ulx, xres, xskew, uly, yskew, yres = s2_ds.GetGeoTransform()
        
        # Calculate lower right corner
        lrx = ulx + (s2_ds.RasterXSize * xres)
        lry = uly + (s2_ds.RasterYSize * yres)
        
        print("Sentinel-2 Tile Extent:")
        print(f"  Upper Left:  ({ulx:.2f}, {uly:.2f})")
        print(f"  Lower Right: ({lrx:.2f}, {lry:.2f})")
        print(f"  Size: {s2_ds.RasterXSize} x {s2_ds.RasterYSize} pixels")
    else:
        print("❌ Could not open Sentinel-2 GeoTIFF")
except Exception as e:
    print(f"⚠ Error: {e}")

### Step 3: Extract CORINE Using Sentinel-2 Extent

In [ ]:
# Extract CORINE map using S2 tile coordinates
corine_output = "/p/scratch/training2600/your_username/corine_data/corine_aligned_tile.tif"

try:
    if s2_ds:
        # Use gdal_translate via Python API
        ds_corine = gdal.Open(corine_reproj, 0)
        
        if ds_corine:
            # Translate (clip) to match S2 extent
            gdal.Translate(
                corine_output,
                ds_corine,
                projWin=[ulx, uly, lrx, lry],  # [ulx, uly, lrx, lry]
                xRes=10,  # Match S2 resolution (10m)
                yRes=10,
                resampleAlg='nearest'
            )
            print(f"✓ CORINE extracted and aligned to Sentinel-2 tile")
            print(f"  Output: {corine_output}")
        else:
            print("❌ Could not open reprojected CORINE")
except Exception as e:
    print(f"❌ Extraction failed: {e}")

### Loading LUCAS Data

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, box
import matplotlib.pyplot as plt

# Load LUCAS points from CSV (download from Eurostat)
# https://ec.europa.eu/eurostat/web/lucas/data/primary-data

lucas_csv = "/p/scratch/training2600/your_username/lucas_data/lucas_2018.csv"

try:
    # Load LUCAS data
    lucas_df = pd.read_csv(lucas_csv)
    
    # Filter for 2018 and convert to GeoDataFrame
    lucas_2018 = lucas_df[lucas_df['SURVEY_YEAR'] == 2018].copy()
    
    # Create geometry from coordinates
    geometry = [Point(xy) for xy in zip(lucas_2018['TH_LONG'], lucas_2018['TH_LAT'])]
    lucas_gdf = gpd.GeoDataFrame(lucas_2018, geometry=geometry, crs='EPSG:4326')
    
    print(f"✓ Loaded {len(lucas_gdf)} LUCAS points from 2018")
    print(f"\nAvailable land cover fields:")
    print(lucas_gdf.columns.tolist()[:15])
    
except FileNotFoundError:
    print("⚠ LUCAS file not found. Download from:")
    print("  https://ec.europa.eu/eurostat/web/lucas/data/primary-data")
    lucas_gdf = None
except Exception as e:
    print(f"⚠ Error loading LUCAS data: {e}")
    lucas_gdf = None

In [ ]:
## Part 5: Extract Training Patches

Extract 3x3 pixel patches around LUCAS ground truth points for model training.

### Filter LUCAS Points Within Tile Extent

In [ ]:
# Filter LUCAS points that fall within our Sentinel-2 tile extent
# First, get extent from a preprocessed Sentinel-2 GeoTIFF

s2_tile_path = "/p/scratch/training2600/your_username/sentinel2_geotiff/S2A_example.tif"

try:
    s2_tile = gdal.Open(s2_tile_path, 0)
    
    if s2_tile and lucas_gdf is not None:
        # Get projection
        projection = s2_tile.GetProjection()
        
        # Transform LUCAS points to match S2 projection
        lucas_projected = lucas_gdf.to_crs(projection)
        
        # Get tile extent
        geotransform = s2_tile.GetGeoTransform()
        ulx, xres, xskew, uly, yskew, yres = geotransform
        lrx = ulx + (s2_tile.RasterXSize * xres)
        lry = uly + (s2_tile.RasterYSize * yres)
        
        # Create bounding box
        tile_bbox = box(min(ulx, lrx), min(uly, lry), max(ulx, lrx), max(uly, lry))
        
        # Filter LUCAS points within tile
        lucas_in_tile = lucas_projected[lucas_projected.geometry.within(tile_bbox)].copy()
        
        print(f"✓ Found {len(lucas_in_tile)} LUCAS points within tile extent")
        
        if len(lucas_in_tile) > 0:
            print(f"\nFirst few points:")
            print(lucas_in_tile[['POINT_ID']].head())
        else:
            print("⚠ No LUCAS points in this tile. Try a different tile or region.")
    else:
        print("❌ Could not load S2 tile or LUCAS data")
        lucas_in_tile = None
        
except Exception as e:
    print(f"⚠ Error: {e}")
    print("Note: Update S2 tile path to match your preprocessed data")
    lucas_in_tile = None

### Patch Extraction Function

In [ ]:
def extract_patch(ds, lon, lat, geotransform, patch_size=3):
    """
    Extract a patch around a geographic point
    
    Parameters:
    -----------
    ds : gdal.Dataset
        Input raster dataset
    lon, lat : float
        Geographic coordinates (in dataset CRS)
    geotransform : tuple
        GDAL geotransform
    patch_size : int
        Size of patch (e.g., 3 for 3x3)
    
    Returns:
    --------
    numpy.ndarray : Patch data (patch_size, patch_size, n_bands) or None
    """
    ulx, xres, xskew, uly, yskew, yres = geotransform
    
    # Convert geographic coordinates to pixel coordinates
    px = int((lon - ulx) / xres)
    py = int((lat - uly) / yres)
    
    # Calculate patch bounds
    half_patch = patch_size // 2
    px_start = px - half_patch
    py_start = py - half_patch
    
    # Check bounds
    if (px_start < 0 or py_start < 0 or 
        px_start + patch_size >= ds.RasterXSize or 
        py_start + patch_size >= ds.RasterYSize):
        return None
    
    # Extract patch for all bands
    n_bands = ds.RasterCount
    patch = np.zeros((patch_size, patch_size, n_bands))
    
    for i in range(n_bands):
        band = ds.GetRasterBand(i + 1)
        data = band.ReadAsArray(px_start, py_start, patch_size, patch_size)
        patch[:, :, i] = data
    
    return patch

print("✓ Patch extraction function defined")

### Extract Patches for All Points

In [ ]:
# Extract patches for all LUCAS points in the tile
patches = []
labels = []
point_ids = []

if s2_tile is not None and lucas_in_tile is not None and len(lucas_in_tile) > 0:
    geotransform = s2_tile.GetGeoTransform()
    
    for idx, point in lucas_in_tile.iterrows():
        lon, lat = point.geometry.x, point.geometry.y
        
        # Extract patch
        patch = extract_patch(s2_tile, lon, lat, geotransform, patch_size=3)
        
        if patch is not None:
            patches.append(patch)
            labels.append(point.get('LC1', 'Unknown'))  # Land Cover label
            point_ids.append(point.get('POINT_ID', idx))
    
    print(f"✓ Extracted {len(patches)} patches")
    if patches:
        print(f"  Patch shape: {patches[0].shape}")
        print(f"\nLabel distribution:")
        print(pd.Series(labels).value_counts()[:10])
else:
    print("⚠ No patches extracted - check S2 tile and LUCAS data")

## Part 6: Visualize and Save Training Data

Visualize extracted patches and save for model training.

In [ ]:
# Visualize first 5 patches
if patches:
    fig, axes = plt.subplots(1, min(5, len(patches)), figsize=(15, 3))
    
    if len(patches) == 1:
        axes = [axes]
    
    for i in range(min(5, len(patches))):
        patch = patches[i]
        
        # Assume first 3 bands are RGB (B2, B3, B4)
        # Normalize for display
        rgb = patch[:, :, :3].copy().astype(float)
        rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
        
        axes[i].imshow(rgb)
        axes[i].set_title(f"LC: {labels[i]}\nID: {point_ids[i]}")
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠ No patches to visualize")

### Save Patches for Training

---

## Summary

This notebook covered the complete preprocessing and patch extraction pipeline:

1. **Reading Rasters**: Opening and inspecting GeoTIFF files with GDAL
2. **Processing S2 Data**: Converting Sentinel-2 SAFE archives to GeoTIFF
3. **Loading Ground Truth**: Reading LUCAS points and filtering by tile extent
4. **Extracting Patches**: Creating 3x3 pixel patches around ground truth locations
5. **Saving Data**: Preparing training data for model development

The extracted patches with labels are ready for use in **Lab 4** model training.

---

## What's Next?

### Before Moving to Lab 4

**1. Verify Extracted Patches**
   - Check that training data file is saved and accessible
   - Verify patch shapes and label distributions
   - Ensure sufficient samples for training

**2. Prepare for Model Training**
   - Understand patch structure: (N, 3, 3, 4) where N=num_patches
   - Labels: Land cover class from LUCAS
   - Consider data augmentation strategies

### Next Lab: Lab 4 - Understanding Transformers

In **Lab 4**, you'll:
- Build a PyTorch dataset loader from saved patches
- Learn transformer architecture for remote sensing
- Train your first classification model
- Evaluate model performance

### Data Pipeline Recap

```python
# Lab 3.1 produces:
Sentinel-2 ZIP files (SAFE format)
    ↓
# Lab 3.2 processes:
Extract bands from SAFE archives
Convert to GeoTIFF
Extract 3x3 patches from LUCAS locations
    ↓
# Lab 4 trains:
Load patches as PyTorch tensors
Build and train transformer model
Save trained weights
```

---

## Resources & References

- **GDAL Documentation**: https://gdal.org/
- **Sentinel-2 Product Specification**: https://sentinels.copernicus.eu/web/sentinel/missions/sentinel-2
- **LUCAS Dataset**: https://ec.europa.eu/eurostat/web/lucas/data/primary-data
- **GeoTIFF Format**: https://www.ogc.org/standards/geotiff

---

## Troubleshooting & FAQ

**Q: How do I handle large Sentinel-2 tiles?**
- Use gdal.SetConfigOption('GDAL_CACHEMAX', 1024) to increase memory cache
- Process tiles in chunks if needed
- Store intermediate results efficiently

**Q: My patches have mostly the same label - is this expected?**
- LUCAS points are sparse - not all areas have ground truth
- Consider using multiple overlapping tiles
- Balance classes before training

**Q: Can I extract larger patches (e.g., 5x5)?**
- Yes! Change `patch_size=3` to `patch_size=5` in the function
- This gives 50m x 50m patches at 10m resolution
- Larger patches may capture more context

**Q: How do I process multiple tiles in parallel?**
- Use HPC job arrays: `#SBATCH --array=0-10`
- Loop over all downloaded tiles in the input directory
- Save results with unique filenames per tile

---

**Course Contact**: Refer to course materials for instructor email and office hours  
**Last Updated**: February 2026